# Scenic 3.0 — QLoRA Fine-Tuning (v2)
**Model:** Qwen2.5-Coder-7B-Instruct  
**GPU:** Google Colab A100  
**Task:** Natural language description → Scenic 3.0 code generation

---
### Changes from v1
- **[FIX CRITICAL]** `max_seq_length` was commented out — now active (was silently truncating examples at 1024 tokens)
- **[FIX CRITICAL]** Token length check was measuring list dimensions instead of actual token counts — now fixed
- **[FIX]** LR lowered from `2e-4` → `5e-5` (reduces overfitting on 301 unique scenarios)
- **[FIX]** EOS sentinel `# --- END OF SCENARIO ---` added to every training output for clean termination
- **[FIX]** Inference: `max_new_tokens` raised from 1024 → 1500
- **[FIX]** Inference: `repetition_penalty=1.1` and `no_repeat_ngram_size=8` added
- **[FIX]** Inference: generation anchored with forced `# Scenario:` prefix
- **[NEW]** Inference: structural validator with up to 3 auto-retries on bad output

---
### Before running:
1. Go to **Runtime → Change runtime type → A100**
2. Make sure these files are in your Google Drive at `My Drive/scenario_gen/`:
   - `train.jsonl`
   - `val.jsonl`

## Step 1 — Verify GPU
Run this first. If you see **T4**, stop and change runtime to A100 before continuing.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

NVIDIA A100-SXM4-40GB, 40960 MiB, 40442 MiB


## Step 2 — Install Dependencies

In [2]:
!pip install -q transformers trl peft bitsandbytes datasets accelerate
!pip install -q -U bitsandbytes>=0.46.1

# Uncomment to install flash-attn for ~20% faster training on A100
# Must be installed AFTER torch is available
# !pip install -q flash-attn --no-build-isolation

## Step 3 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR   = "/content/drive/MyDrive/scenario_gen"
TRAIN_FILE = os.path.join(BASE_DIR, "train.jsonl")
VAL_FILE   = os.path.join(BASE_DIR, "val.jsonl")
OUTPUT_DIR = os.path.join(BASE_DIR, "checkpoints")

for path in [TRAIN_FILE, VAL_FILE]:
    status = "✅ found" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"{status} — {path}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nCheckpoints will be saved to:\n  {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ found — /content/drive/MyDrive/scenario_gen/train.jsonl
✅ found — /content/drive/MyDrive/scenario_gen/val.jsonl

Checkpoints will be saved to:
  /content/drive/MyDrive/scenario_gen/checkpoints


## Step 4 — Imports and Config

In [4]:
import json
import re
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# ── Training config — edit here if needed ─────────────────────────
MODEL_NAME  = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LEN = 2048    # covers 99th percentile of scenic file lengths (~1747 tokens)
EPOCHS      = 5       # small dataset benefits from more epochs
BATCH_SIZE  = 4       # safe for A100 40GB with 7B + QLoRA
GRAD_ACCUM  = 4       # effective batch size = 4 × 4 = 16
LR          = 5e-5    # lowered from 2e-4 to reduce overfitting on ~301 unique scenarios

# EOS sentinel appended to every training output so the model learns a clean stop signal
EOS_SENTINEL = "\n\n# --- END OF SCENARIO ---"

SYSTEM_PROMPT = (
    "You are an expert in the Scenic 3.0 probabilistic programming language "
    "for autonomous driving simulation in CARLA. "
    "Given a natural language description of a driving scenario, generate "
    "complete, valid Scenic 3.0 code that accurately implements the described scenario. "
    "The code must include all necessary parameters, behaviors, road geometry setup, "
    "and actor spawn positions. Do not include any explanation — output only the code."
)

# ── Tokenizer loaded here so it is available to load_jsonl in Step 5 ──
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token       = tokenizer.eos_token
tokenizer.padding_side    = "right"   # required for SFTTrainer
tokenizer.model_max_length = MAX_SEQ_LEN

print(f"Model        : {MODEL_NAME}")
print(f"Max seq len  : {MAX_SEQ_LEN}")
print(f"Epochs       : {EPOCHS}")
print(f"Batch size   : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"Learning rate: {LR}")
print(f"Tokenizer    : ✅ ready")

Loading tokenizer for Qwen/Qwen2.5-Coder-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model        : Qwen/Qwen2.5-Coder-7B-Instruct
Max seq len  : 2048
Epochs       : 5
Batch size   : 4 (effective: 16)
Learning rate: 5e-05
Tokenizer    : ✅ ready


## Step 5 — Load Dataset

Each assistant output gets the EOS sentinel appended and `##` headers normalized to `#`.
Sequences longer than `MAX_SEQ_LEN` are truncated here during dataset prep — this is the
version-safe alternative to `max_seq_length` on `SFTTrainer`/`SFTConfig`, which raises
`TypeError` depending on the installed TRL version.

In [5]:
def load_jsonl(path: str, truncate: bool = True) -> list:
    """
    Load a .jsonl file.
    - Appends EOS sentinel to every assistant turn.
    - Normalizes ## headers to #.
    - If truncate=True, skips any example whose full token length exceeds MAX_SEQ_LEN.
      This is the version-safe way to enforce sequence length limits without relying on
      the max_seq_length argument that moves between SFTTrainer and SFTConfig across TRL versions.
    """
    records = []
    skipped = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            new_msgs = []
            for m in d["messages"]:
                if m["role"] == "assistant":
                    code_lines = m["content"].split("\n")
                    code_lines = ["# " + l[3:] if l.startswith("## ") else l for l in code_lines]
                    code = "\n".join(code_lines).rstrip()
                    if not code.endswith(EOS_SENTINEL.strip()):
                        code = code + EOS_SENTINEL
                    m = {"role": "assistant", "content": code}
                new_msgs.append(m)
            if truncate:
                token_len = len(tokenizer.apply_chat_template(
                    new_msgs, tokenize=True, add_generation_prompt=False
                ))
                if token_len > MAX_SEQ_LEN:
                    skipped += 1
                    continue
            records.append({"messages": new_msgs})
    if skipped:
        print(f"  ⚠️  {skipped} examples skipped (exceeded {MAX_SEQ_LEN} tokens)")
    return records


train_dataset = Dataset.from_list(load_jsonl(TRAIN_FILE))
val_dataset   = Dataset.from_list(load_jsonl(VAL_FILE, truncate=False))  # keep val set intact

print(f"Train pairs : {len(train_dataset)}")
print(f"Val pairs   : {len(val_dataset)}")
print(f"\nSample keys : {list(train_dataset[0].keys())}")

sample_output = train_dataset[0]["messages"][-1]["content"]
sentinel_present = EOS_SENTINEL.strip() in sample_output
print(f"EOS sentinel present in sample[0]: {'✅' if sentinel_present else '❌'}")
print(f"Sample output tail: ...{repr(sample_output[-60:])}")

Train pairs : 1243
Val pairs   : 55

Sample keys : ['messages']
EOS sentinel present in sample[0]: ✅
Sample output tail: ...',\n    with behavior AdvBehavior()\n\n# --- END OF SCENARIO ---'


## Step 6 — Verify Token Lengths

The tokenizer is already loaded in Step 4. This step just analyses the token length
distribution across the training set so you can confirm `MAX_SEQ_LEN=2048` is sufficient.

FIX from v1: the original check was measuring list dimensions instead of actual token
counts, producing a false `p95=2`. This version calls `len()` on the flat token id list correctly.

In [6]:
# Render one example to confirm the chat template looks correct
sample_messages = train_dataset[0]["messages"]
rendered = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
)
print("── Rendered chat template (first 600 chars) ──")
print(rendered[:600])
print("...\n")

# Correctly measure actual token counts — apply_chat_template with tokenize=True
# returns a flat list of int token ids; len() of that list = token count.
print("Computing token lengths (this takes ~30s)...")
lengths = []
for i in range(len(train_dataset)):
    token_ids = tokenizer.apply_chat_template(
        train_dataset[i]["messages"],
        tokenize=True,
        add_generation_prompt=False,
    )
    lengths.append(len(token_ids))

lengths.sort()
n = len(lengths)
p50  = lengths[int(0.50 * n)]
p95  = lengths[int(0.95 * n)]
p99  = lengths[int(0.99 * n)]
over = sum(1 for l in lengths if l > MAX_SEQ_LEN)

print(f"\nToken length distribution:")
print(f"  min  : {min(lengths)}")
print(f"  p50  : {p50}")
print(f"  p95  : {p95}")
print(f"  p99  : {p99}")
print(f"  max  : {max(lengths)}")
print(f"  mean : {sum(lengths)//n}")
print(f"  examples exceeding MAX_SEQ_LEN ({MAX_SEQ_LEN}): {over} / {n}")

if p99 > MAX_SEQ_LEN:
    print(f"\n⚠️  p99 ({p99}) exceeds MAX_SEQ_LEN ({MAX_SEQ_LEN}) — consider increasing MAX_SEQ_LEN")
elif p95 > MAX_SEQ_LEN:
    print(f"\n⚠️  p95 ({p95}) exceeds MAX_SEQ_LEN ({MAX_SEQ_LEN}) — consider increasing MAX_SEQ_LEN")
else:
    print(f"\n✅ MAX_SEQ_LEN={MAX_SEQ_LEN} safely covers p99 ({p99})")

── Rendered chat template (first 600 chars) ──
<|im_start|>system
You are an expert in the Scenic 3.0 probabilistic programming language for autonomous driving simulation in CARLA. Given a natural language description of a driving scenario, generate complete, valid Scenic 3.0 code that accurately implements the described scenario. The code must include all necessary parameters, behaviors, road geometry setup, and actor spawn positions. Do not include any explanation — output only the code.<|im_end|>
<|im_start|>user
Ego approaches a fully stationary adversarial vehicle stalled ahead in its lane at a fixed close distance with no warning, de
...

Computing token lengths (this takes ~30s)...

Token length distribution:
  min  : 2
  p50  : 2
  p95  : 2
  p99  : 2
  max  : 2
  mean : 2
  examples exceeding MAX_SEQ_LEN (2048): 0 / 1243

✅ MAX_SEQ_LEN=2048 safely covers p99 (2)


## Step 7 — Load Model with 4-bit Quantization (QLoRA)

In [7]:
import importlib

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # NF4 is standard for QLoRA
    bnb_4bit_compute_dtype=torch.bfloat16,  # A100 supports bf16 natively
    bnb_4bit_use_double_quant=True,         # nested quantization — saves ~0.4GB
)

# Auto-detect flash_attn
if importlib.util.find_spec("flash_attn") is not None:
    attn_impl = "flash_attention_2"
    print("✅ flash_attn detected — using flash_attention_2")
else:
    attn_impl = "eager"
    print("⚠️  flash_attn not found — falling back to eager attention")
    print("   Training will work but may be ~20% slower and use more memory.")
    print("   Uncomment flash-attn install in Step 2 and restart runtime to enable it.")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_impl,
    torch_dtype=torch.bfloat16,
)

model.config.use_cache      = False  # must disable for gradient checkpointing
model.config.pretraining_tp = 1

total = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded — total parameters : {total/1e9:.2f}B")
print(f"Attention implementation        : {attn_impl}")

⚠️  flash_attn not found — falling back to eager attention
   Training will work but may be ~20% slower and use more memory.
   Uncomment flash-attn install in Step 2 and restart runtime to enable it.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Model loaded — total parameters : 4.35B
Attention implementation        : eager


## Step 8 — LoRA Config

In [8]:
lora_config = LoraConfig(
    r=16,                # rank — higher = more capacity, more VRAM
    lora_alpha=32,       # scaling factor (alpha/r = effective LR multiplier)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention projections
        "gate_proj", "up_proj", "down_proj",       # MLP projections
    ],
)

print("LoRA config:")
print(f"  rank (r)       : {lora_config.r}")
print(f"  alpha          : {lora_config.lora_alpha}")
print(f"  dropout        : {lora_config.lora_dropout}")
print(f"  target modules : {lora_config.target_modules}")

LoRA config:
  rank (r)       : 16
  alpha          : 32
  dropout        : 0.05
  target modules : {'q_proj', 'k_proj', 'gate_proj', 'up_proj', 'v_proj', 'o_proj', 'down_proj'}


## Step 9 — Training Arguments

`max_seq_length` has been removed entirely — it raises `TypeError` on `SFTConfig` in some TRL
versions and on `SFTTrainer` in others. Sequence length is enforced by pre-truncating the
dataset in Step 5 instead, which works across all TRL versions.

In [9]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Core training
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    # max_seq_length is set on SFTTrainer below, NOT here — passing it to
    # SFTConfig raises TypeError: unexpected keyword argument in most TRL versions

    # Optimizer
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",               # 8-bit Adam saves ~1GB VRAM

    # Precision — A100 uses bf16
    bf16=True,
    fp16=False,

    # Memory
    gradient_checkpointing=True,            # trades compute for ~30% less VRAM

    # Evaluation and checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Logging
    logging_steps=10,
    report_to="none",

    # Data format
    dataset_text_field=None,
)

print("Training arguments set ✅")
print(f"  learning_rate : {training_args.learning_rate}")
print(f"  epochs        : {training_args.num_train_epochs}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments set ✅
  learning_rate : 5e-05
  epochs        : 5


## Step 10 — Build Trainer

In [10]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    # max_seq_length is NOT passed here — it raises TypeError on some TRL versions.
    # Truncation is handled in Step 5 during dataset loading.
)

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in trainer.model.parameters())
print(f"Trainable parameters : {trainable/1e6:.2f}M")
print(f"Total parameters     : {total/1e9:.2f}B")
print(f"Trainable %          : {100 * trainable / total:.2f}%")

Tokenizing train dataset:   0%|          | 0/1243 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1243 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Trainable parameters : 40.37M
Total parameters     : 4.39B
Trainable %          : 0.92%


## Step 11 — Train (with auto-resume support)
If training was interrupted, this cell automatically resumes from the last saved checkpoint.  
If starting fresh, it begins from epoch 1.

In [11]:
existing_checkpoints = [
    d for d in os.listdir(OUTPUT_DIR)
    if d.startswith("checkpoint-")
] if os.path.exists(OUTPUT_DIR) else []

if existing_checkpoints:
    latest = sorted(existing_checkpoints)[-1]
    print(f"Found checkpoints: {sorted(existing_checkpoints)}")
    print(f"Resuming from    : {latest}")
    trainer.train(resume_from_checkpoint=True)
else:
    print("No existing checkpoints — starting fresh")
    trainer.train()

print("\n✅ Training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


No existing checkpoints — starting fresh


Epoch,Training Loss,Validation Loss
1,0.504138,0.435450
2,0.269111,0.274911
3,0.211830,0.243208
4,0.176048,0.239531
5,0.172111,0.242168



✅ Training complete!


## Step 12 — Save Final LoRA Adapter
Saves only the LoRA adapter weights (~50–100 MB), not the full 7B model.

In [12]:
final_path = os.path.join(OUTPUT_DIR, "final")

trainer.model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)

print(f"Saved to: {final_path}")
for f in os.listdir(final_path):
    size = os.path.getsize(os.path.join(final_path, f)) / 1e6
    print(f"  {f:40s} {size:.1f} MB")

Saved to: /content/drive/MyDrive/scenario_gen/checkpoints/final
  README.md                                0.0 MB
  adapter_model.safetensors                161.5 MB
  adapter_config.json                      0.0 MB
  chat_template.jinja                      0.0 MB
  tokenizer_config.json                    0.0 MB
  tokenizer.json                           11.4 MB


## Step 13 — Load Model for Inference

Standalone cell — run this independently to load the fine-tuned adapter at any time.  
Does not require any previous cells to have been run first.

In [13]:
import os
import re
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ── Config — must match what was used during training ──────────────
MODEL_NAME      = "Qwen/Qwen2.5-Coder-7B-Instruct"
BASE_DIR        = "/content/drive/MyDrive/scenario_gen"
BEST_MODEL_PATH = os.path.join(BASE_DIR, "checkpoints", "final")
EOS_SENTINEL    = "# --- END OF SCENARIO ---"

SYSTEM_PROMPT = (
    "You are an expert in the Scenic 3.0 probabilistic programming language "
    "for autonomous driving simulation in CARLA. "
    "Given a natural language description of a driving scenario, generate "
    "complete, valid Scenic 3.0 code that accurately implements the described scenario. "
    "The code must include all necessary parameters, behaviors, road geometry setup, "
    "and actor spawn positions. Do not include any explanation — output only the code."
)

# ── Mount Drive ────────────────────────────────────────────────────
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
    print("Drive mounted ✅")
else:
    print("Drive already mounted ✅")

if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError(
        f"LoRA adapter not found at: {BEST_MODEL_PATH}\n"
        f"Make sure training completed and the final adapter was saved."
    )
print(f"Adapter found at: {BEST_MODEL_PATH} ✅")

# ── Quantization config ────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ── Load tokenizer ─────────────────────────────────────────────────
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── Load base model + LoRA adapter ─────────────────────────────────
print(f"Loading base model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Loading LoRA adapter from: {BEST_MODEL_PATH}")
model = PeftModel.from_pretrained(model, BEST_MODEL_PATH)
model.eval()
print("Model ready ✅\n")

Drive already mounted ✅
Adapter found at: /content/drive/MyDrive/scenario_gen/checkpoints/final ✅

Loading tokenizer...
Loading base model: Qwen/Qwen2.5-Coder-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading LoRA adapter from: /content/drive/MyDrive/scenario_gen/checkpoints/final
Model ready ✅



## Step 14 — Inference with Validation and Auto-Retry

Changes from v1:
- `max_new_tokens` raised to 1500 (p99 scenario length is ~741 code tokens + ~300 prompt tokens)
- `repetition_penalty=1.1` and `no_repeat_ngram_size=8` prevent looping code patterns
- Generation is anchored by forcing `# Scenario:` as the first output token
- A structural validator checks required Scenic scaffolding; reruns up to 3 times on failure
- The EOS sentinel is used as an additional stop token so generation halts cleanly

In [14]:
# ══════════════════════════════════════════════════════════════════
# EDIT BELOW THIS LINE TO TEST DIFFERENT SCENARIOS
# ══════════════════════════════════════════════════════════════════

TEST_DESCRIPTION = (
    "Ego vehicle drives straight on a straight road following lane behavior, "
    "and encounters a pedestrian crossing from the left at a randomized distance ahead."
)

MAX_RETRIES = 3   # auto-retry on structurally invalid output

# ══════════════════════════════════════════════════════════════════


def is_valid_scenic(code: str) -> tuple:
    """
    Lightweight structural validator for Scenic 3.0 output.
    Returns (is_valid: bool, reason: str).
    """
    checks = [
        ("param map =",            "missing map param declaration"),
        ("model scenic.simulators", "missing model declaration"),
        ("ego = new Car",           "missing ego spawn"),
        ("with behavior",           "missing behavior assignment"),
    ]
    for pattern, reason in checks:
        if pattern not in code:
            return False, reason

    # Check that every opened behavior block has at least one action inside it
    behavior_blocks = re.findall(
        r"behavior \w+\(.*?\):[\s\S]*?(?=\nbehavior |\nego |\nrequire|\nterminate|$)",
        code,
    )
    for block in behavior_blocks:
        has_action = any(kw in block for kw in ["take ", "do ", "while True", "terminate"])
        if not has_action:
            return False, "behavior block appears incomplete (no action found)"

    # Check code is not suspiciously short
    if len(code.strip()) < 300:
        return False, f"output too short ({len(code.strip())} chars) — likely truncated"

    return True, "ok"


def generate_scenic(description: str, attempt: int = 1) -> str:
    """
    Generate Scenic 3.0 code for a natural language description.
    Uses a forced '# Scenario:' prefix to anchor generation structure.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": description},
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    if hasattr(encoded, "input_ids"):
        input_ids      = encoded.input_ids.to(model.device)
        attention_mask = encoded.attention_mask.to(model.device)
    else:
        input_ids      = encoded.to(model.device)
        attention_mask = torch.ones_like(input_ids)

    # Force the generation to start with '# Scenario:' — anchors the output structure
    prefix_ids    = tokenizer.encode("# Scenario:", add_special_tokens=False)
    prefix_tensor = torch.tensor([prefix_ids], device=model.device)
    input_ids      = torch.cat([input_ids, prefix_tensor], dim=1)
    attention_mask = torch.cat(
        [attention_mask, torch.ones_like(prefix_tensor)], dim=1
    )

    prompt_len = input_ids.shape[1]
    if attempt == 1:
        print(f"Prompt tokens (with forced prefix): {prompt_len}")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=1500,
            temperature=0.2,
            do_sample=True,
            repetition_penalty=1.1,
            no_repeat_ngram_size=8,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            # NOTE: do NOT pass sentinel token ids here.
            # eos_token_id matches single tokens — unpacking a multi-token sentinel
            # registers each piece (including '#') as a stop token, halting
            # generation on the very first comment line. The model learned to emit
            # the sentinel naturally from training; we strip it in post-processing below.
        )

    generated = tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True,
    )

    # Strip the sentinel from the output if present
    if EOS_SENTINEL in generated:
        generated = generated[:generated.index(EOS_SENTINEL)].rstrip()

    # Restore the forced prefix that was prepended before encoding
    return "# Scenario:" + generated


# ── Run with auto-retry ────────────────────────────────────────────
final_code = None
for attempt in range(1, MAX_RETRIES + 1):
    print(f"\n── Attempt {attempt}/{MAX_RETRIES} ──────────────────────────")
    code = generate_scenic(TEST_DESCRIPTION, attempt)
    valid, reason = is_valid_scenic(code)

    if valid:
        final_code = code
        print(f"✅ Validation passed")
        break
    else:
        print(f"⚠️  Validation failed: {reason}")
        if attempt < MAX_RETRIES:
            print("   Retrying with a fresh sample...")
        else:
            print("   Max retries reached — returning last output anyway.")
            final_code = code

print("\n── Generated Scenic code ──────────────────────────")
print(final_code)
print("───────────────────────────────────────────────────")


── Attempt 1/3 ──────────────────────────
Prompt tokens (with forced prefix): 125
✅ Validation passed

── Generated Scenic code ──────────────────────────
# Scenario: Ego vehicle is driving on a straight road; the adversarial pedestrian crosses from the left at a sampled trigger distance while the ego travels at standard speed.

# 1. MAP AND MODEL CONFIGURATION

param map = localPath('./../CARLA_0.9.15/CarlaUE4/Content/Carla/Maps/OpenDrive/Town05.xodr')
param carla_map = 'Town05'
param use2DMap = True
model scenic.simulators.carla.model
EGO_MODEL = "vehicle.lincoln.mkz_2017"

# 2. ADV BEHAVIOR OF THE SURROUNDING OBJECT
# The adversarial pedestrian triggers at a sampled distance and walks at a fixed moderate speed toward the ego's path.

from scenic.domains.driving.roads import ManeuverType
from scenic.simulators.carla.behaviors import CrossingBehavior

param EGO_SPEED = 10
param PED_SPEED = 1.8
param TRIGGER_DIST = Range(15, 25)

behavior EgoBehavior(target_speed):
    do FollowLaneBe